# MDS — ILP (PuLP)

$$\min \sum_v x_v \quad s.t. \quad \forall v:\; \sum_{u\in N[v]} x_u \ge 1,\; x_v\in\{0,1\}$$

Solver CBC vía PuLP. Baseline **clásico exacto** (poliédrico).


In [ ]:
import itertools
import json
import math
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np

import pulp


In [ ]:
nodes = ["Mario", "Sarah", "Raul", "Ana", "Enrique", "Saul"]
edges = [("Mario", "Sarah"), ("Mario", "Raul"), ("Mario", "Ana"),
         ("Enrique", "Sarah"), ("Enrique", "Raul"), ("Saul", "Ana")]
idx = {v: i for i, v in enumerate(nodes)}
n = len(nodes)
adj = {i: set() for i in range(n)}
for a, b in edges:
    adj[idx[a]].add(idx[b])
    adj[idx[b]].add(idx[a])
closed = {v: sorted(adj[v] | {v}) for v in range(n)}

def bits_cost_payments(bits):
    return int(sum(bits))

def is_dominating(bits):
    return all(any(bits[u] for u in closed[v]) for v in range(n))

def decode(bits):
    return [nodes[i] for i in range(n) if bits[i]]

# Óptimo exacto (referencia compartida)
best_cost, best_bits = n + 1, None
for k in range(2 ** n):
    bits = [(k >> i) & 1 for i in range(n)]
    if is_dominating(bits):
        c = bits_cost_payments(bits)
        if c < best_cost:
            best_cost, best_bits = c, bits
OPTIMAL_COST = float(best_cost)
OPTIMAL_SOL = decode(best_bits)
print("Óptimo MDS:", OPTIMAL_SOL, "coste =", OPTIMAL_COST)

def resolve_metrics_dir() -> Path:
    for base in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        root = base / "Notebooks" / "Optimization" / "MinimumDominatingSetOptimization"
        if root.is_dir():
            d = root / "metrics"
            d.mkdir(parents=True, exist_ok=True)
            return d
    d = Path.cwd().resolve() / "Notebooks" / "Optimization" / "MinimumDominatingSetOptimization" / "metrics"
    d.mkdir(parents=True, exist_ok=True)
    return d

def save_mds_metrics(payload: dict, filename: str):
    path = resolve_metrics_dir() / filename
    path.write_text(json.dumps(payload, indent=2))
    print("Saved:", path)
    return path


## Modelo y resolución


In [ ]:
t0 = time.perf_counter()
prob = pulp.LpProblem("MDS", pulp.LpMinimize)
x = pulp.LpVariable.dicts("x", range(n), cat="Binary")
prob += pulp.lpSum(x[i] for i in range(n))
for v in range(n):
    prob += pulp.lpSum(x[u] for u in closed[v]) >= 1
status = prob.solve(pulp.PULP_CBC_CMD(msg=False))
elapsed = time.perf_counter() - t0
bits = [1 if pulp.value(x[i]) > 0.5 else 0 for i in range(n)]
sol = decode(bits)
c = bits_cost_payments(bits)
feasible = is_dominating(bits)
success = feasible and abs(c - OPTIMAL_COST) < 1e-9
print("Status:", pulp.LpStatus[status], "sol:", sol, "coste:", c)

payload = {
    "problem": "mds",
    "solver": "ilp",
    "family": "classical",
    "protocol": {"n_runs": 1, "method": "pulp_cbc_covering_ilp"},
    "instance": {"nodes": nodes, "edges": [list(e) for e in edges]},
    "quality": {
        "optimal_cost": OPTIMAL_COST,
        "optimal_solution": OPTIMAL_SOL,
        "cost_mean": float(c),
        "cost_best": float(c),
        "gap_mean": float(c - OPTIMAL_COST),
    },
    "success_rate": float(success),
    "time_s": {"mean": float(elapsed), "std": 0.0, "total": float(elapsed)},
    "effort": {"milp_status": pulp.LpStatus[status], "n_binary_vars": n},
    "runs": [{"seed": 0, "solution": sol, "cost": float(c), "feasible": feasible, "success": bool(success), "time_s": float(elapsed)}],
}
save_mds_metrics(payload, "ilp_mds.json")
